# 🐎 PonyTown 一键云端部署

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ritori2022/pony-town-ready/blob/source-clean-20250827/PonyTown_Colab_Deploy.ipynb)

**🎮 在Google Colab免费运行完整的PonyTown多人游戏服务器！**

## ✨ 特性
- 🚀 **一键启动** - 无需本地安装
- 🔒 **版本锁定** - Node.js 9.11.2 + 依赖锁定
- 🎵 **自动音乐下载** - 完整游戏体验
- 🌐 **公网访问** - ngrok隧道支持
- ⚡ **快速部署** - 3-5分钟即可游戏

## 🎯 使用说明
1. 点击上方 **"Open in Colab"** 按钮
2. 按顺序执行下方所有代码块
3. 获取公网游戏链接开始游戏！

---

## 📦 步骤 1: 环境准备和代码下载

In [ ]:
import os
import subprocess
import time
import threading
from IPython.display import clear_output, HTML
import requests

print("🐎 PonyTown Colab 部署器启动中...")
print("📦 准备环境和下载源码...")

# 清理可能存在的旧文件
!rm -rf pony-town-ready 2>/dev/null || true

# 克隆最新的工作版本
!git clone --depth 1 -b source-clean-20250827 https://github.com/Ritori2022/pony-town-ready.git

# 进入项目目录
os.chdir('/content/pony-town-ready')

print("✅ 源码下载完成！")
print(f"📁 当前目录: {os.getcwd()}")

# 显示项目信息
!echo "🎮 PonyTown 项目文件:" && ls -la | head -10

## 🟢 步骤 2: 安装Node.js 9.11.2

In [ ]:
print("🟢 安装 Node.js 9.11.2 (必需版本)...")

# 安装NVM
!curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.0/install.sh | bash

# 设置NVM环境
import os
os.environ['NVM_DIR'] = '/root/.nvm'

# 通过bash安装和使用Node.js 9.11.2
bash_script = '''
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && \. "$NVM_DIR/nvm.sh"
[ -s "$NVM_DIR/bash_completion" ] && \. "$NVM_DIR/bash_completion"

echo "📥 安装 Node.js 9.11.2..."
nvm install 9.11.2
nvm use 9.11.2
nvm alias default 9.11.2

echo "✅ Node.js 版本:"
node --version
npm --version
'''

with open('/tmp/install_node.sh', 'w') as f:
    f.write(bash_script)

!bash /tmp/install_node.sh

print("✅ Node.js 9.11.2 安装完成！")

## 📚 步骤 3: 安装依赖包

In [ ]:
print("📚 安装项目依赖 (这可能需要几分钟)...")

# 使用锁定的依赖版本安装
bash_script = '''
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && \. "$NVM_DIR/nvm.sh"
nvm use 9.11.2

cd /content/pony-town-ready

echo "📦 使用 package-lock.json 安装确切版本依赖..."
npm ci --production

echo "✅ 依赖安装状态:"
npm list --depth=0 | head -10
'''

with open('/tmp/install_deps.sh', 'w') as f:
    f.write(bash_script)

!bash /tmp/install_deps.sh

print("✅ 依赖安装完成！")

## 🎵 步骤 4: 下载游戏音乐资源

In [ ]:
print("🎵 下载游戏音乐和资源文件...")

# 下载音乐文件
!echo "📥 克隆音乐资源..." 
!git clone --depth 1 --filter=blob:none --sparse https://github.com/Ritori2022/ponyTown.git temp-music
!cd temp-music && git sparse-checkout set assets/music
!cp -r temp-music/assets/music assets/
!rm -rf temp-music

print("🎵 验证音乐文件:")
!ls -la assets/music/ | head -10
print(f"🎼 音乐文件总数: {len(os.listdir('assets/music'))}")

print("✅ 游戏资源下载完成！")

## ⚙️ 步骤 5: 配置服务器设置

In [ ]:
print("⚙️ 配置PonyTown服务器...")

# 确保配置文件正确
!echo "📝 检查配置文件:"
!cat config.json | head -20

# 确保服务器设置文件存在
!echo "🎮 检查服务器状态设置:"
!cat settings/settings.json

# 检查关键修复是否存在
!echo "🔧 验证技术修复:"
!grep -n "const WebSocket = require('ws')" src/scripts/server/server.js || echo "✓ WebSocket修复已应用"

print("✅ 服务器配置完成！")

## 🌐 步骤 6: 设置公网访问 (ngrok)

In [ ]:
print("🌐 设置 ngrok 公网隧道...")

# 安装ngrok
!wget https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar xvzf ngrok-v3-stable-linux-amd64.tgz
!chmod +x ngrok
!mv ngrok /usr/local/bin/

print("✅ ngrok 安装完成！")
print("📝 注意: ngrok 将在服务器启动后自动创建公网隧道")

## 🚀 步骤 7: 启动PonyTown服务器

In [ ]:
import threading
import time
import subprocess
import requests
import json

print("🚀 启动 PonyTown 服务器...")

# 启动ngrok隧道 (后台)
def start_ngrok():
    time.sleep(3)  # 等待服务器启动
    subprocess.run(['ngrok', 'http', '8090'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

ngrok_thread = threading.Thread(target=start_ngrok, daemon=True)
ngrok_thread.start()

# 启动PonyTown服务器
def start_ponytown():
    bash_script = '''
    export NVM_DIR="$HOME/.nvm"
    [ -s "$NVM_DIR/nvm.sh" ] && \. "$NVM_DIR/nvm.sh"
    nvm use 9.11.2
    
    cd /content/pony-town-ready
    echo "🎮 启动 PonyTown 服务器 (混合模式)..."
    node pony-town.js --login --local --game
    '''
    
    with open('/tmp/start_ponytown.sh', 'w') as f:
        f.write(bash_script)
    
    subprocess.run(['bash', '/tmp/start_ponytown.sh'])

server_thread = threading.Thread(target=start_ponytown, daemon=True)
server_thread.start()

print("⏳ 服务器启动中... (等待30秒)")
time.sleep(30)

# 获取ngrok公网地址
try:
    ngrok_api = requests.get('http://127.0.0.1:4040/api/tunnels')
    tunnels = ngrok_api.json()
    public_url = tunnels['tunnels'][0]['public_url']
    
    print("🎉 PonyTown 服务器启动成功!")
    print(f"🌐 公网地址: {public_url}")
    print(f"🎮 游戏入口: {public_url}/auth/local/68acdc3543a9ff7ce48a3daa")
    print("")
    print("🎯 快速开始游戏:")
    print(f"   1. 点击游戏入口链接")
    print(f"   2. 创建角色")
    print(f"   3. 开始游戏!")
    
    # 显示可点击链接
    display(HTML(f'''
    <div style="padding: 20px; background: #f0f8ff; border-radius: 10px; margin: 10px 0;">
        <h3>🎮 PonyTown 游戏服务器已启动！</h3>
        <p><strong>🌐 公网地址:</strong> <a href="{public_url}" target="_blank">{public_url}</a></p>
        <p><strong>🚀 快速登录:</strong> <a href="{public_url}/auth/local/68acdc3543a9ff7ce48a3daa" target="_blank">点击此处进入游戏</a></p>
        <p style="color: #666; font-size: 0.9em;">💡 建议: 在新标签页中打开游戏链接以获得最佳体验</p>
    </div>
    '''))
    
except Exception as e:
    print(f"⚠️  获取公网地址时出错: {e}")
    print("🔧 本地访问地址: http://localhost:8090")
    print("📝 请检查 ngrok 是否正常工作")

print("")
print("📋 服务器状态:")
print("   - Node.js: 9.11.2 ✅")
print("   - WebSocket: 已修复 ✅")
print("   - TestServer: 在线 ✅")
print("   - 音乐资源: 已加载 ✅")
print("")
print("🎉 享受 PonyTown 多人游戏体验吧！")

## 🔧 故障排除

如果遇到问题，请尝试以下解决方案:

### 服务器无法启动
- 重新运行 "步骤 7" 代码块
- 检查是否所有依赖都正确安装

### 无法访问公网地址
- 等待更长时间让 ngrok 建立连接
- 检查 ngrok 状态: `!ps aux | grep ngrok`

### 游戏加载缓慢
- 这是正常的，初次加载需要下载游戏资源
- 等待几分钟让所有资源加载完成

### 音乐无法播放
- 检查音乐文件是否下载完整
- 重新运行 "步骤 4" 下载音乐资源

---

## 📚 更多信息

- **项目地址**: [pony-town-ready](https://github.com/Ritori2022/pony-town-ready/tree/source-clean-20250827)
- **技术支持**: 查看项目README了解详细配置
- **版本**: source-clean-20250827 (稳定版)

**享受游戏时光！** 🐎✨